In [1]:
import warnings
import copy
from itertools import combinations

import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mutual_info_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from mrmr import mrmr_classif
from SurrogateAssistedWrapperFeatureSelection import SAGA, select_top_k_features
from hsic import dHSIC


In [2]:
data_folder = "data/"
data_list = ["data_combat_corrected.csv", "adata_with_labels.csv"]
df_list = [pd.read_csv(data_folder + i) for i in data_list]
warnings.filterwarnings("ignore")


# Feature selection algorithms

## Adaptive ElasticNet

In [3]:
def get_x_y_feature(df):
    x = df.drop(columns=["label"])
    return x, df["label"], x.columns


def preprocess(df):
    X, y, feature_names = get_x_y_feature(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_scaled = scaler.transform(X)

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train)
    beta_init = ridge.coef_

    gamma = 1.0
    weights = 1 / (np.abs(beta_init) + 1e-3) ** gamma
    X_train_weighted = X_train_scaled / weights
    return X_train_weighted, y_train, weights


def rank_features(df, model, feature_num, weights):
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]
    feature_names = X.columns
    coef = model.coef_ / weights
    ranking = pd.DataFrame(
        {"Feature": feature_names, "Coefficient": coef, "Abs_Coefficient": np.abs(coef)}
    )
    ranking = ranking.sort_values(by="Abs_Coefficient", ascending=False)

    top_features = ranking.head(feature_num)

    selected_features = top_features["Feature"].values
    X_selected = X[selected_features]

    return pd.concat([X_selected, y], axis=1)


def adaptive_elasticnet(df, feature_num=30):
    X_train_weighted, y_train, weights = preprocess(df)
    enet = ElasticNetCV()
    enet.fit(X_train_weighted, y_train)

    return rank_features(df, enet, feature_num, weights)


## Adaptive Lasso

In [4]:
def adaptive_lasso(df, feature_num=30):
    X_train_weighted, y_train, weights = preprocess(df)
    lasso = LassoCV()
    lasso.fit(X_train_weighted, y_train)
    return rank_features(df, lasso, feature_num, weights)


# mRMR

In [5]:
def mRMR(df, feature_num=30):
    X = df.drop(columns=["label"])
    y = df["label"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    selected_features = mrmr_classif(
        X=X_train, y=y_train, K=feature_num, show_progress=False
    )
    return df[selected_features + ["label"]].copy()


## VCSDFS

In [6]:
def vcsdfs(df, feature_num=30):
    X, y, feature_names = get_x_y_feature(df)
    feature_names = np.array(feature_names)
    X = X.values
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X_train, X_test = train_test_split(
        X,
        test_size=0.2,
    )
    XX = X_train.T @ X_train
    XXX = XX @ XX

    rho = 0.1
    n_iter = 50
    n, d = X_train.shape
    W = np.random.rand(d, feature_num)
    ones_dd = np.ones((d, d))
    eps = 1e-12
    for it in range(n_iter):
        numW = XXX @ W + rho * W
        A = XX @ W
        denW = A @ W.T @ A + rho * (ones_dd @ W)
        fracW = numW / (denW + eps)
        fracW = np.power(fracW, 0.25)
        W = W * fracW

    score = np.sum(W * W, axis=1)
    index = np.argsort(score)[::-1]
    selected_features = feature_names[index[:feature_num]]
    return df[selected_features.tolist() + ["label"]]


## IGRF-RFE

In [7]:
RFE_PATIENCE_LIMIT = 3
MLP_HIDDEN_DIM = 128
MLP_OUTPUT_DIM = 1
MLP_EPOCHS = 20
MLP_BATCH_SIZE = 128
MLP_LEARNING_RATE = 0.001
MLP_EARLY_STOPPING_PATIENCE = 3


class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.layer_stack = nn.Sequential(
            nn.Linear(input_dim, MLP_HIDDEN_DIM),
            nn.BatchNorm1d(MLP_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(MLP_HIDDEN_DIM, MLP_HIDDEN_DIM),
            nn.BatchNorm1d(MLP_HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(MLP_HIDDEN_DIM, MLP_OUTPUT_DIM),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.layer_stack(x)


def train_and_evaluate_mlp(X_train_sub, y_train_sub, X_val_sub, y_val_sub):
    input_dim = X_train_sub.shape[1]
    model = MLP(input_dim)
    loss_fn = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=MLP_LEARNING_RATE)

    train_dataset = TensorDataset(
        torch.FloatTensor(X_train_sub), torch.FloatTensor(y_train_sub.reshape(-1, 1))
    )
    val_dataset = TensorDataset(
        torch.FloatTensor(X_val_sub), torch.FloatTensor(y_val_sub.reshape(-1, 1))
    )

    train_loader = DataLoader(train_dataset, batch_size=MLP_BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=MLP_BATCH_SIZE)

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(MLP_EPOCHS):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                y_pred_val = model(X_batch)
                val_loss += loss_fn(y_pred_val, y_batch).item()
        val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1

        if patience_counter >= MLP_EARLY_STOPPING_PATIENCE:
            break

    if best_model_state:
        model.load_state_dict(best_model_state)
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            y_pred_val = model(X_batch)
            predicted = (y_pred_val > 0.5).float()
            all_preds.extend(predicted.numpy())
            all_labels.extend(y_batch.numpy())

    return accuracy_score(all_labels, all_preds)


def get_rf_selected_features(X_train, y_train, feature_names_list, threshold=0.55):
    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
    rf.fit(X_train, y_train)
    scores = rf.feature_importances_
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min())
    df = pd.DataFrame({"Feature": feature_names_list, "Score": scores_normalized})
    return df[df["Score"] >= threshold]["Feature"].tolist()


def get_ig_selected_features(X_train, y_train, feature_names_list, threshold=0.55):
    scores = mutual_info_classif(X_train, y_train)
    scores_normalized = (scores - scores.min()) / (scores.max() - scores.min())
    df = pd.DataFrame({"Feature": feature_names_list, "Score": scores_normalized})
    return df[df["Score"] >= threshold]["Feature"].tolist()


def igrf_rfe(df, feature_num=30):
    NUM_EXPERIMENTS = 3
    X, y, feature_names = get_x_y_feature(df)
    feature_names_list = feature_names.tolist()

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.25, stratify=y_train_val
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    y_train_arr = y_train.values
    y_val_arr = y_val.values
    y_test_arr = y_test.values

    rf_selected = get_rf_selected_features(
        X_train_scaled, y_train_arr, feature_names_list
    )
    ig_selected = get_ig_selected_features(
        X_train_scaled, y_train_arr, feature_names_list
    )
    initial_feature_names = list(set(ig_selected).union(set(rf_selected)))
    initial_feature_indices = [
        feature_names_list.index(f) for f in initial_feature_names
    ]
    current_features = list(initial_feature_indices)
    optimal_feature_subset = list(initial_feature_indices)

    X_tr_sub = X_train_scaled[:, current_features]
    X_val_sub = X_val_scaled[:, current_features]
    initial_scores = [
        train_and_evaluate_mlp(X_tr_sub, y_train_arr, X_val_sub, y_val_arr)
        for _ in range(NUM_EXPERIMENTS)
    ]
    best_overall_accuracy = np.mean(initial_scores)

    patience = 0
    iteration = 0
    while len(current_features) > feature_num:
        iteration += 1
        performance_dict = {}

        for idx in current_features:
            temp_features = [f for f in current_features if f != idx]

            X_tr_sub = X_train_scaled[:, temp_features]
            X_val_sub = X_val_scaled[:, temp_features]

            scores = [
                train_and_evaluate_mlp(X_tr_sub, y_train_arr, X_val_sub, y_val_arr)
                for _ in range(NUM_EXPERIMENTS)
            ]

            performance_dict[idx] = np.mean(scores)

        feature_to_remove = max(performance_dict, key=performance_dict.get)
        best_local_accuracy = performance_dict[feature_to_remove]
        current_features.remove(feature_to_remove)

        if best_local_accuracy > best_overall_accuracy:
            best_overall_accuracy = best_local_accuracy
            patience = 0
        else:
            patience += 1

    final_selected_features = [feature_names_list[i] for i in current_features]

    selected_feature_names = final_selected_features
    columns_to_keep = selected_feature_names + ["label"]
    return df[columns_to_keep]


## SAGA-KNN

In [8]:
def saga_knn(df, feature_num=30):
    X, y, feature_names = get_x_y_feature(df)
    X = X.to_numpy()
    y = y.to_numpy()
    feature_names_list = feature_names.tolist()
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    knn_params = {
        "n_neighbors": 5,
        "weights": "uniform",
        "algorithm": "auto",
        "n_jobs": -1,
    }

    best_mask, selected_indices, test_accuracy = SAGA(
        X_train,
        y_train,
        X_test,
        y_test,
        knn_params,
    )

    indices = select_top_k_features(
        best_mask, X_train, y_train, knn_params, k=feature_num, cv_folds=5
    )
    feature_names = df.columns[:-1].tolist()
    selected_feature_names = [feature_names[i] for i in indices]
    return df[selected_feature_names + [df.columns[-1]]]


# Triangular Informative Data Function (TIDF)

In [9]:
def redundancy_rate_nmi(df, bins=10, strategy="uniform", drop_na=True):
    df_numeric = df.select_dtypes(include=[np.number])
    if drop_na:
        df_numeric = df_numeric.dropna()

    d = df_numeric.shape[1]
    if d < 2:
        return 0.0, 0

    X = df_numeric.values
    discretizer = KBinsDiscretizer(n_bins=bins, encode="ordinal", strategy=strategy)
    X_discrete = discretizer.fit_transform(X).astype(int)

    sim_sum = 0.0
    for i, j in combinations(range(d), 2):
        sim = mutual_info_score(X_discrete[:, i], X_discrete[:, j])
        sim_sum += sim

    rr = (2.0 / (d * (d - 1))) * sim_sum
    rr = np.clip(rr, 0.0, 1.0)
    return rr


def normalized_mutual_information(x, y):
    n = len(x)
    contingency = np.zeros((x.max() + 1, y.max() + 1), dtype=np.float64)
    for xi, yi in zip(x, y):
        contingency[xi, yi] += 1
    contingency /= n
    px = contingency.sum(axis=1)
    py = contingency.sum(axis=0)

    hx = -np.sum(px[px > 0] * np.log(px[px > 0]))
    hy = -np.sum(py[py > 0] * np.log(py[py > 0]))
    if hx == 0 or hy == 0:
        return 0

    mi = 0.0
    for i in range(contingency.shape[0]):
        for j in range(contingency.shape[1]):
            pxy = contingency[i, j]
            if pxy > 0 and px[i] > 0 and py[j] > 0:
                mi += pxy * np.log(pxy / (px[i] * py[j]))

    nmi = mi / np.sqrt(hx * hy)
    return float(np.clip(nmi, 0.0, 1.0))


def representation_entropy(
    df, target_col="label", log_base="natural", normalize=True, use_full_dims=False
):
    X = df.drop(columns=[target_col])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    cov_matrix = np.cov(X_scaled, rowvar=False)
    eigenvalues = np.linalg.eigvalsh(cov_matrix)
    eigenvalues = np.maximum(eigenvalues, 0)
    total_variance = np.sum(eigenvalues)
    if total_variance == 0:
        return 0.0, 0.0
    lambda_hat = eigenvalues / total_variance
    lambda_hat_pos = lambda_hat[lambda_hat > 0]
    if log_base == "natural":
        log_func = np.log
    elif log_base == "2":
        log_func = np.log2
    else:
        raise ValueError("log_base must be 'natural' or '2'")
    entropy_R = -np.sum(lambda_hat_pos * log_func(lambda_hat_pos))

    if normalize:
        if use_full_dims:
            d = X.shape[1]
            entropy_max = log_func(d) if d > 0 else 1
        else:
            d = len(lambda_hat_pos)
            entropy_max = log_func(d) if d > 0 else 1
        entropy_norm = entropy_R / entropy_max if entropy_max > 0 else 0
    else:
        entropy_norm = None
    return entropy_R, entropy_norm


def hilbert_schmidt_independence(df):
    X, y, _ = get_x_y_feature(df)
    X = X.to_numpy()
    y = y.to_numpy()
    return dHSIC(X, y)


def TID(df, W):
    rr = redundancy_rate_nmi(df)
    _, re = representation_entropy(df)
    hsic = hilbert_schmidt_independence(df)
    h = W[2] * re
    b = W[0] * hsic + W[1] * (1 - rr)
    if h == 0:
        return b
    if b == 0:
        return h
    return h * b / 2


# WHFDL2

In [10]:
def WHFDL2(df, fs_algs, W, MaxIter=10, feature_num=30):
    N = len(fs_algs)
    LTIDF = [0] * N
    Run = [0] * N
    TIDF_0 = 0
    reward_history = dict([(i.__name__, list()) for i in fs_algs])
    print("First run:")
    for i, alg in enumerate(fs_algs):
        print(alg.__name__, end="\t\t")
        new_df = alg(df, feature_num)
        TIDF = TID(new_df, W)
        print("TIDF:", TIDF)
        LTIDF[i] = TIDF
        TIDF_0 = TIDF

    RLalpha = 0
    print()
    print("Main loop:")
    for t in range(MaxIter):
        chN = np.random.random()
        if chN <= RLalpha:
            i = LTIDF.index(max(LTIDF))
        else:
            i = np.random.randint(0, N)
        print(f"{t+1}/{MaxIteration}", fs_algs[i].__name__, end="\t")

        new_df = fs_algs[i](df, feature_num)
        TIDF = TID(new_df, W)
        print("TIDF:", TIDF)
        LTIDF[i] += 1
        if TIDF == TIDF_0:
            LTIDF[i] = max(0, LTIDF[i] - ((sum(LTIDF) - LTIDF[i]) / N))
        RLalpha += 1 / MaxIteration
        TIDF_0 = TIDF
        Run[i] += 1

        for n, alg in enumerate(fs_algs):
            reward_history[alg.__name__].append(LTIDF[n])

    return LTIDF.index(max(LTIDF)), reward_history, Run


In [11]:
MaxIteration = 10
feature_num = 30
w = [1, 1, 1]
fs_algs = [
    adaptive_elasticnet,
    adaptive_lasso,
    mRMR,
    vcsdfs,
    igrf_rfe,
    saga_knn,
]

for i, df in enumerate(df_list):
    print(f"Running WHFDL2 on {data_list[i]}")
    best_fs_alg_ind, reward_history, Run = WHFDL2(
        df, fs_algs, w, MaxIteration, feature_num
    )
    print("\nResults:")
    for j, alg in enumerate(fs_algs):
        print(
            alg.__name__,
            "\tRun:",
            Run[j],
            "\treward:",
            reward_history[alg.__name__][-1],
        )
    print("\nBest FS algorithm:", fs_algs[best_fs_alg_ind].__name__)
    print()


Running WHFDL2 on data_combat_corrected.csv
First run:
adaptive_elasticnet		TIDF: 0.3949259958660806
adaptive_lasso		TIDF: 0.40111408326379533
mRMR		TIDF: 0.3924784466135041
vcsdfs		TIDF: 0.4367151584943344
igrf_rfe		TIDF: 0.4230155272969654
saga_knn		TIDF: 0.41898872578514

Main loop:
1/10 vcsdfs	TIDF: 0.4367151584943344
2/10 igrf_rfe	TIDF: 0.36185260387285745
3/10 adaptive_lasso	TIDF: 0.40829792811860205
4/10 vcsdfs	TIDF: 0.4367151584943344
5/10 adaptive_elasticnet	TIDF: 0.4084909145175602
6/10 vcsdfs	TIDF: 0.4367151584943344
7/10 adaptive_elasticnet	TIDF: 0.4006766400534403
8/10 vcsdfs	TIDF: 0.4367151584943344
9/10 vcsdfs	TIDF: 0.4367151584943344
10/10 vcsdfs	TIDF: 0.4367151584943344

Results:
adaptive_elasticnet 	Run: 2 	reward: 2.394925995866081
adaptive_lasso 	Run: 1 	reward: 1.4011140832637954
mRMR 	Run: 0 	reward: 0.3924784466135041
vcsdfs 	Run: 6 	reward: 4.426540898885839
igrf_rfe 	Run: 1 	reward: 1.4230155272969653
saga_knn 	Run: 0 	reward: 0.41898872578514

Best FS algorith